In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

In [9]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [10]:
from pinecone import Pinecone, ServerlessSpec
index_name="hybrid-search-langchain"

# initialise pinecone client
pc = Pinecone(api_key=api_key)

# Create index

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension = 384, # dimension of dense model, 384 bcoz using hugginface allmini sentence transformer
        metric = 'dotproduct', # for sparse value
        spec = ServerlessSpec(cloud = 'aws',region = "us-east-1"),

    )

In [11]:
index = pc.Index(index_name)
index

In [13]:
# vector embeddings

from langchain_huggingface import HuggingFaceEmbeddings

os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7572.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [14]:
# for sparse matrix
from pinecone_text.sparse import BM25Encoder # uses TFIDF by default

bm25_encoder = BM25Encoder().default()
bm25_encoder

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rishabh/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [15]:
sentences=[
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

# tfidf values on sentences
bm25_encoder.fit(sentences)

# store values to a json file
bm25_encoder.dump("bm25_values.json")

# load bm25_encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 88.75it/s]


In [16]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder, index=index)
# this retriver supports both 

In [17]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x3247b16c0>, index=<pinecone.db_data.index.Index object at 0x10f5d3250>)

In [18]:
retriever.add_texts(
      [  "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
      ]
)

100%|██████████| 1/1 [00:02<00:00,  2.42s/it]


In [32]:
retriever.invoke("Which city did I visit in 2022?")

[Document(metadata={'score': 0.490436673}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.355388373}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.339708447}, page_content='In 2021, I visited New Orleans')]